In [3]:
import pandas as pd
import re
import ftfy

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

df = pd.read_csv('crm_contacts_raw.csv', engine='pyarrow')

In [4]:
# Cleaning Lead ID column
df['Lead ID'] = df['Lead ID'].str.strip()

def clean_lead_id(cell):
    if cell is None:
        return None
    s = str(cell).strip().upper()

    if re.fullmatch(r"[A-Z]-\d{5}", s):
        return s

    m = re.fullmatch(r"([A-Z])(\d{5})", s)
    if m:
        return f"{m[1]}-{m[2]}"

    if re.fullmatch(r"\d{4}", s):
        return f"X-0{s}"

    m = re.fullmatch(r"[A-Z]{4}(\d{4})", s)
    if m:
        return f"L-0{m[1]}"

    return None

df["Lead ID"] = df["Lead ID"].apply(clean_lead_id).replace('NAN', pd.NA)

In [18]:
# Cleaning Full Name column
df['Full Name'] = (df['Full Name']
    .str.strip()
    .str.replace(
        {
            r'\s{2,}': ' ',
            r'\b(?:Mr|Mrs|Ms|Miss|Dr|Prof)\b\.?\s*': '',
            r'[\s,]+\b(?:Jr|Sr|II|III|IV|PhD|MD|Esq)\b\.?\s*$': ''
        }
    , regex=True))


def safe_fix(x):
    if isinstance(x, str):
        return ftfy.fix_text(x)
    return x

df['Full Name'] = df['Full Name'].apply(safe_fix)

def main_clean_name(cell):
    if pd.isna(cell):
        return pd.NA
    cell = cell.strip()
    
    # Swap
    if ',' in cell:
        parts = cell.split(',')
        last_name, first_name = parts[0].strip(), parts[1].strip()
        cell = f"{first_name} {last_name}"
    

df['Full Name'].head(500)

0                   emma chen
1             Lars De la Cruz
2          Christopher Wilson
3              Jürgen Schmidt
4                Tariq Wilson
5               Emma Sørensen
6               Johnson, Anna
7            łukasz rodriguez
8                       Mateo
9             Katherine Smith
10             Michael Harris
11                  john chen
12                Søren Singh
13             tariq campbell
14               Jones, Tariq
15                sarah allen
16             Alexander Chen
17              Daniel Murphy
18                   Lars Lee
19           Elizabeth Wilson
20               Nadia Martin
21                 NOAH NOWAK
22               Priya Ivanov
23             Jürgen Jackson
24             Sørensen, Mary
25               Olivia Singh
26             Amélie Schmidt
27             Davis, Michael
28                Robert Khan
29                 Kim, Priya
30     Katherine Van der Berg
31             MICHAEL HARRIS
32           William Williams
33        